# Figure: N&#8322;O flux time series and cumulative emissions

Half-hourly and daily aggregated N&#8322;O fluxes per treatment, plus cumulative
emissions with a correlation-aware uncertainty estimate (random flux error,
gap-filling residuals and u\* threshold ensemble).

**Input** `../data/raw/fluxes_meteo_mgmt.parquet`
**Output** `../figures/Timeseries_*.png`, `../figures/Timeseries+cumulative_*.png`


# LOAD LIBRARIES

In [ ]:
import numpy as np
import pandas as pd # pandas allows to do a lot of basic data manipulation such as loading csv data files
from datetime import datetime
import matplotlib.pyplot as plt # matplotlib is a basic ploting library of Python
import matplotlib.transforms as mtransforms
import matplotlib.dates as mdates
from diive.core.io.files import load_parquet

# LOAD DATA

In [ ]:
# Load data
filename = "../data/raw/fluxes_meteo_mgmt.parquet"
data = load_parquet(filename)

# Select the gapfilled fluxes
fluxes = ['NEE', 'FN2O', 'FCH4']

# Observed fluxes (OBS)
obs_fluxes = [f'{flux}_L3.3_CUT_50_QCF0_{trt}' for flux in fluxes for trt in ['parcelA', 'parcelB', 'footprint']]

# Gap-filled (QCF)
gf_fluxes = [f'{flux}_L3.3_CUT_50_QCF_{trt}_gfXGBoost' for flux in fluxes for trt in ['parcelA', 'parcelB', 'footprint']]

# Extreme ustar scenarios
ustar_extremes = [f'{flux}_L3.3_CUT_{cut}_QCF_{trt}_gfXGBoost' for flux in fluxes for trt in ['parcelA', 'parcelB', 'footprint'] for cut in ['16', '84']]

# Flux uncertainty (pointwise; kept for reference, no longer used for the
# cumulative bands in plot_cum -- see that function for why)
flux_unc = [f'{col}_total_unc' for col in gf_fluxes]

# Uncertainty INPUTS needed to rebuild a correlation-aware cumulative band
# later (see plot_cum). Requires the upstream merge to have been built with
# merge_gapfill_results(..., store_uncertainty_inputs=True).
flux_sigmaObs = [f'{col}_sigmaObs' for col in gf_fluxes]
flux_sigmaResid = [f'{col}_sigmaResid' for col in gf_fluxes]
flux_ens = [c for c in data.columns if any(c.startswith(f'{col}_ens') for col in gf_fluxes)]

missing_ens = [col for col in gf_fluxes if not any(c.startswith(f'{col}_ens') for c in data.columns)]
if missing_ens:
    raise KeyError(
        f"No '<col>_ens##' columns found for: {missing_ens}. The upstream merge needs "
        f"to have used merge_gapfill_results(..., store_uncertainty_inputs=True) for "
        f"the cumulative ensemble uncertainty to be reconstructable here."
    )

# Flags corresponding to gap-filled (QCF)
gf_flags = [f'FLAG_{c}_ISFILLED' for c in gf_fluxes]

# Keep only relevant columns
data = data[gf_fluxes + ustar_extremes + gf_flags + obs_fluxes + flux_unc
            + flux_sigmaObs + flux_sigmaResid + flux_ens].copy()

# Create a df with filter out days without any observed data
data_filtered = data.copy()
MIN_OBS_PER_DAY = 1 
for col in gf_fluxes:
    flag_col = f'FLAG_{col}_ISFILLED'
    flag = data_filtered[flag_col]
    day_index = data_filtered.index.floor('D')
    obs_count_by_day = (flag == 0).groupby(day_index).sum()
    keep_day = obs_count_by_day >= MIN_OBS_PER_DAY
    keep_rows = day_index.map(keep_day).fillna(False)
    data_filtered.loc[~keep_rows, col] = np.nan

# Set time interval
start_date = '2023-11-01'
end_date = '2025-06-05'
df = data[start_date:end_date].copy()
df_filtered = data_filtered[start_date:end_date].copy()

# Plot fluxes
indata = df[gf_fluxes]
indata.plot(x_compat=True, subplots=True, figsize=(12, 2*len(indata.columns)))
plt.show()

df


# PLOTS

## CONFIGS

In [ ]:
from plot_config import EVENT_LINES, LABELS, COLORS, PERIODS, add_event_lines

print(EVENT_LINES)
print(LABELS)
print(COLORS)
print(PERIODS)

## TIME SERIES

Aggregate

In [ ]:
df_agg = pd.DataFrame()
df_agg_filtered = pd.DataFrame()
incols = gf_fluxes + obs_fluxes
for col in incols:
    df_agg_filtered[col + '_mean'] = df_filtered[col].resample('D').mean()
    df_agg_filtered[col + '_sd'] = df_filtered[col].resample('D').std()
    df_agg[col + '_mean'] = df[col].resample('D').mean()
    df_agg[col + '_sd'] = df[col].resample('D').std()

df_agg

function for single gas

In [ ]:
def plot_flux_timeseries(df_in, gas, ax, period_name, start_extent=None, end_extent=None, mgmt=True):

    # Select the period
    start, end = PERIODS[period_name]
    start_ts = pd.to_datetime(start)
    end_ts = pd.to_datetime(end)
    indat = df_in[start_ts:end_ts].copy()
    # Columns to plot
    incols = [c for c in df_in if c.startswith(gas)]
    # Define means and stdevs
    mA_col = next((c for c in incols if 'parcelA' in c and c.endswith('_mean')), None)
    mA = indat[mA_col]
    sA_col = next((c for c in incols if 'parcelA' in c and c.endswith('_sd')), None)
    sA = indat[sA_col]
    mB_col = next((c for c in incols if 'parcelB' in c and c.endswith('_mean')), None)
    mB = indat[mB_col]
    sB_col = next((c for c in incols if 'parcelB' in c and c.endswith('_sd')), None)
    sB = indat[sB_col]

    # Legend labels for this period
    label_A, label_B = LABELS['A'], LABELS['B']

    # Plot line with filled area between +- stdev
    ax.fill_between(indat.index, mA - sA, mA + sA, alpha=0.25, color=COLORS['A'])
    ax.plot(indat.index, mA, color=COLORS['A'], label=label_A)
    ax.fill_between(indat.index, mB - sB, mB + sB, alpha=0.25, color=COLORS['B'])
    ax.plot(indat.index, mB, color=COLORS['B'], label=label_B)
    # Draw event lines and labels only above the top plot

    # If available, use start_extent/end_extent to limit the range of event lines drawn and for the x-axis limits.
    if start_extent is not None:
        start_ts = pd.to_datetime(start_extent)
        end_ts = pd.to_datetime(end_extent)
    # Add event lines and labels
    add_event_lines(ax, start_ts=start_ts, end_ts=end_ts, mgmt=mgmt, height=1.01, x_spacing_days=3)
    # x-axis formatting
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    secax = ax.secondary_xaxis('bottom')
    secax.xaxis.set_major_locator(mdates.YearLocator())
    secax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    secax.tick_params(axis='x', length=0, pad=30)
    # Set x-limits
    ax.set_xlim(start_ts, end_ts)
    # y axis settings 
    ax.axhline(0, color='grey', linestyle='--', lw=1)
    ax.yaxis.set_major_locator(plt.MaxNLocator(4))
    # Legend
    ax.legend(loc='upper right', bbox_to_anchor=(0.95, 1), frameon=False)
    # Remove top spine
    ax.spines['top'].set_visible(False)

### FN2O

In [ ]:
indat = df_agg[[c for c in df_agg.columns if 'XGBoost' in c]]

fig, ax = plt.subplots(figsize=(12, 6))
plot_flux_timeseries(indat, 
                     period_name='wheat_treatment_period', 
                     gas='FN2O',
                     start_extent='2024-03-01',
                     end_extent='2024-07-26', 
                     ax=ax)
ax.set_ylabel('N$_2$O flux\n(nmol m$^{-2}$ s$^{-1}$)')
plt.show()

## CUMULATIVE

In [ ]:
# Convert units
T = 60 * 30  # seconds per 30 min
MW_N2 = 28.014 
MW_C = 12.011

factor_dict = {
    'FN2O' : T * MW_N2 * 1e-9 * 1e-3 * 1e4, # nmol N2O m-2 s-1 → kg N2O-N ha-1 30min-1
    'FCH4': T * MW_C * 1e-9 * 1e-3 * 1e4, # nmol CH4 m-2 s-1 → kg CH4-C ha-1 30min-1
    'NEE': T * MW_C * 1e-6 * 1e-3 * 1e4 # umol CO2 m-2 s-1 → kg CO2-C ha-1 30min-1
}

# Build all columns in a dict first, then construct the DataFrame in one
# shot -- with the ensemble columns added (~20 per gf_flux) this is now a
# few hundred columns, and inserting them one at a time into an existing
# DataFrame fragments its internal block manager (slow, and pandas warns).
kg_ha_cols = {}

for col in gf_fluxes:
    flux = col.split('_')[0]
    kg_ha_cols[col] = df[col] * factor_dict[flux]

# change units for uncertainty columns to kg ha-1 30min-1, so that they can be summed to cumulative uncertainty bands in plot_cum
for col in flux_unc + flux_sigmaObs + flux_sigmaResid + flux_ens:
    flux = col.split('_')[0]
    kg_ha_cols[col] = df[col] * factor_dict[flux]

for col in ustar_extremes:
    flux = col.split('_')[0]
    kg_ha_cols[col] = df[col] * factor_dict[flux]

# Gap-fill flags are 0/1 indicators, not physical quantities -->  no need to convert
for col in gf_flags:
    kg_ha_cols[col] = df[col]

df_kg_ha = pd.DataFrame(kg_ha_cols, index=df.index)
df_kg_ha

### Function

In [ ]:
import re

def _ensemble_cumulative_sigma_series(subset, col_base):
    """
    Correlation-aware structural (ensemble) uncertainty AT EVERY TIMESTEP over `subset`'s rows
    Returns a pd.Series indexed like `subset`, restarted at zero at
    subset's first row -- so slicing to a period BEFORE calling this
    reproduces "uncertainty of that period's own cumulative total", matching
    how the cumulative flux itself is computed in plot_cum.
    """
    ens_cols = sorted(c for c in subset.columns if re.fullmatch(rf"{re.escape(col_base)}_ens\d+", c))
    if not ens_cols:
        raise KeyError(
            f"No '{col_base}_ens##' columns found -- the upstream merge needs to have "
            f"used merge_gapfill_results(..., store_uncertainty_inputs=True) for the "
            f"cumulative ensemble uncertainty to be reconstructable here."
        )
    flag_col = f"FLAG_{col_base}_ISFILLED"
    if flag_col not in subset.columns:
        raise KeyError(f"{flag_col!r} not found; needed to know which rows were gap-filled.")
    is_gap = subset[flag_col].astype(bool).to_numpy()

    preds_ens = subset[ens_cols].to_numpy().T            # (K, T)
    masked = np.where(is_gap[None, :], preds_ens, 0.0)   # zero out observed rows
    cum_paths = np.nancumsum(masked, axis=1)             # (K, T) -- each fold's running total
    return pd.Series(np.std(cum_paths, axis=0, ddof=1), index=subset.index)


def _cumulative_uncertainty(df_in, col_base, start_ts, end_ts):
    """
    Correlation-aware cumulative uncertainty for `col_base` over
    [start_ts, end_ts]. sigma_obs (measured rows) and sigma_resid
    (gap-filled rows) are assumed independent row-to-row, so a plain
    quadrature cumsum is used for those; sigma_ens is not assumed as independent,
    so it uses the member-path treatment above instead.
    """
    sub = df_in.loc[start_ts:end_ts]
    is_gap = sub[f"FLAG_{col_base}_ISFILLED"].astype(bool)
    sigma_obs = sub[f"{col_base}_sigmaObs"]
    sigma_resid = sub[f"{col_base}_sigmaResid"]

    sq_obs = (sigma_obs.where(~is_gap, 0.0).fillna(0.0) ** 2).cumsum()
    sq_res = (sigma_resid.where(is_gap, 0.0).fillna(0.0) ** 2).cumsum()
    sens_cum = _ensemble_cumulative_sigma_series(sub, col_base)

    return np.sqrt(sq_obs + sq_res + sens_cum ** 2)


# Function to plot cumulative fluxes for a given period
def plot_cum(df_in, gas, ax, period_name, start_extent=None, end_extent=None, mgmt=True,
              shade_ci=True, ci_z=1.96, show_ustar_lines=False):
    """
    shade_ci:
        True -> draw shaded band of +/- (ci_z * sigma_total), where sigma_total
        combines the propagated flux uncertainty (obs+resid+ens) AND the
        u*-threshold uncertainty in quadrature (Eq. 3 in the methods).
    ci_z:
        1.0 for ~68% band if the underlying sigma is 1-sigma
        1.96 for ~95% band if the underlying sigma is 1-sigma and you want ~95%
    show_ustar_lines:
        True -> also draw the CUT_16/CUT_84 cumulative curves as dashed lines.
        These are independent of shade_ci: the band already reflects u*
        uncertainty either way, this only controls whether the two individual
        scenario curves that generated sigma_ustar are also drawn explicitly.
    """
    
    # Select period
    if period_name is not None:
        start, end = PERIODS[period_name]
        print(f"Cumulative {gas} flux — {period_name}\n{start} to {end}")
    start_ts = pd.to_datetime(start)
    end_ts   = pd.to_datetime(end)

    # Select columns to plot
    flux_cols = [c for c in gf_fluxes if c.startswith(gas)]
    indat = df_in[flux_cols].copy()
    # Map long column names to A/B
    COL_A = next((c for c in indat.columns if 'parcelA' in c), None)
    COL_B = next((c for c in indat.columns if 'parcelB' in c), None)
    # Ustar extremes
    COL_A_USTAR_16 = COL_A.replace('CUT_50', 'CUT_16')
    COL_A_USTAR_84 = COL_A.replace('CUT_50', 'CUT_84')
    COL_B_USTAR_16 = COL_B.replace('CUT_50', 'CUT_16')
    COL_B_USTAR_84 = COL_B.replace('CUT_50', 'CUT_84')

    # Select the period
    indat = indat.loc[start_ts:end_ts].copy()
    ustar = df_in.loc[start_ts:end_ts, [COL_A_USTAR_16, COL_A_USTAR_84, COL_B_USTAR_16, COL_B_USTAR_84]].copy()

    # Cumulative sums (keep gaps from breaking the curve)
    cum = pd.DataFrame(index=indat.index)
    cum["A"] = indat[COL_A].cumsum()
    cum["B"] = indat[COL_B].cumsum()

    # Cumulative u*-threshold scenario curves -- same restart-at-period-start
    # convention as `cum` above, since they're cumsum'd over the same sliced range
    ustar_cum_A16 = ustar[COL_A_USTAR_16].cumsum()
    ustar_cum_A84 = ustar[COL_A_USTAR_84].cumsum()
    ustar_cum_B16 = ustar[COL_B_USTAR_16].cumsum()
    ustar_cum_B84 = ustar[COL_B_USTAR_84].cumsum()

    # Propagated flux uncertainty: correlation-aware for the ensemble/structural
    # term, quadrature for the observation/residual terms -- see
    # _cumulative_uncertainty for why these need different treatment
    prop_unc = pd.DataFrame(index=indat.index)
    prop_unc["A"] = _cumulative_uncertainty(df_in, COL_A, start_ts, end_ts)
    prop_unc["B"] = _cumulative_uncertainty(df_in, COL_B, start_ts, end_ts)

    # u*-threshold uncertainty AT EVERY TIMESTEP: half the range between the
    # lowest and highest of the three CUT_16/CUT_50/CUT_84 cumulative curves
    # (same definition as sigma_ustar in the budgets notebook, just evaluated
    # as a running series here instead of only at the final total)
    ustar_lo_A = pd.concat([ustar_cum_A16, cum["A"], ustar_cum_A84], axis=1).min(axis=1)
    ustar_hi_A = pd.concat([ustar_cum_A16, cum["A"], ustar_cum_A84], axis=1).max(axis=1)
    ustar_lo_B = pd.concat([ustar_cum_B16, cum["B"], ustar_cum_B84], axis=1).min(axis=1)
    ustar_hi_B = pd.concat([ustar_cum_B16, cum["B"], ustar_cum_B84], axis=1).max(axis=1)
    ustar_unc = pd.DataFrame(index=indat.index)
    ustar_unc["A"] = (ustar_hi_A - ustar_lo_A) / 2
    ustar_unc["B"] = (ustar_hi_B - ustar_lo_B) / 2

    # Total uncertainty: propagated flux uncertainty and u*-threshold
    # uncertainty combined in quadrature (Eq. 3)
    cum_unc = pd.DataFrame(index=indat.index)
    cum_unc["A"] = np.sqrt(prop_unc["A"] ** 2 + ustar_unc["A"] ** 2)
    cum_unc["B"] = np.sqrt(prop_unc["B"] ** 2 + ustar_unc["B"] ** 2)

    # Totals (kg N2O-N ha^-1) for this period
    tot_A = float(cum["A"].iloc[-1])
    tot_B = float(cum["B"].iloc[-1])

    # Print totals in the cell
    print(f"Total A: {tot_A:.3f}")
    print(f"Total B: {tot_B:.3f}")

    # Legend labels for this period
    label_A, label_B = LABELS['A'], LABELS['B']

    # Plotting cumulative fluxes
    ax.plot(cum.index, cum["A"], label=label_A, color=COLORS['A'])
    ax.plot(cum.index, cum["B"], label=label_B, color=COLORS['B'])
    # Plotting cumulative fluxes for ustar extremes (dashed lines) -- optional,
    # independent of shade_ci: the band already reflects u* uncertainty either way
    if show_ustar_lines:
        ax.plot(cum.index, ustar_cum_A16, color=COLORS['A'], linestyle='--', alpha=0.5)
        ax.plot(cum.index, ustar_cum_A84, color=COLORS['A'], linestyle='--', alpha=0.5)
        ax.plot(cum.index, ustar_cum_B16, color=COLORS['B'], linestyle='--', alpha=0.5)
        ax.plot(cum.index, ustar_cum_B84, color=COLORS['B'], linestyle='--', alpha=0.5)

    # Shaded uncertainty band (propagated flux unc. + u*-threshold unc., in quadrature)
    if shade_ci:
        z = float(ci_z)
        ax.fill_between(
            cum.index,
            cum["A"] - z * cum_unc["A"],
            cum["A"] + z * cum_unc["A"],
            color=COLORS["A"],
            alpha=0.20,
            linewidth=0
        )
        ax.fill_between(
            cum.index,
            cum["B"] - z * cum_unc["B"],
            cum["B"] + z * cum_unc["B"],
            color=COLORS["B"],
            alpha=0.20,
            linewidth=0
        )

    # If available, use start_extent/end_extent to limit the range of event lines drawn and for the x-axis limits.
    if start_extent is not None:
        start_ts = pd.to_datetime(start_extent)
        end_ts = pd.to_datetime(end_extent)
        
    # Add event lines and labels
    add_event_lines(ax, start_ts=start_ts, end_ts=end_ts, mgmt=mgmt, height=1.01, x_spacing_days=3)

    # x-axis formatting
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.set_xlim(start_ts, end_ts)
    secax = ax.secondary_xaxis('bottom')
    secax.xaxis.set_major_locator(mdates.YearLocator())  # Year ticks
    secax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))  # Format as year
    secax.tick_params(axis='x', length=0, pad=30)  # Add padding for better readability
    ax.legend(frameon=False, bbox_to_anchor=(1, -0.4), loc='lower right')
    # Remove top spine    
    ax.spines['top'].set_visible(False)

### FN2O

In [ ]:
flux = 'FN2O'
period = 'wheat_treatment_period'
fig, ax = plt.subplots(figsize=(10, 6))
plot_cum(df_kg_ha, flux, ax, period_name=period, start_extent='2024-03-01', end_extent='2024-07-26', shade_ci=True, ci_z=1.96)
ax.set_ylabel(flux)
fig.tight_layout()
plt.show()

## COMBINED

In [ ]:
flux = 'FN2O'
period = 'wheat_treatment_period'
start_extent = '2024-03-01'
end_extent = '2024-07-26'

plt.rcParams.update({'font.size': 12})  # Set global font size
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(14, 7.5), gridspec_kw={'hspace': 0.025})
plot_flux_timeseries(df_agg_filtered, gas=flux, ax=axes[0], period_name=period, start_extent=start_extent, end_extent=end_extent)
plot_cum(df_kg_ha, gas=flux, ax=axes[1], period_name=period, start_extent=start_extent, end_extent=end_extent, mgmt=False)
letterx, lettery = 0.001, 0.95
axes[0].text(letterx, lettery, '(a)', transform=axes[0].transAxes, fontweight='bold', verticalalignment='top')
axes[1].text(letterx, lettery, '(b)', transform=axes[1].transAxes, fontweight='bold', verticalalignment='top')
axes[0].set_ylabel('N$_2$O flux\n(nmol m$^{-2}$ s$^{-1}$)')
axes[1].set_ylabel('Cumulative N$_{2}$O flux\n(kg N$_{2}$O-N ha$^{-1}$)')
axes[1].get_legend().remove()
fig.align_ylabels()
fig.savefig(f'../figures/Timeseries+cumulative_{flux}_{period}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
flux = 'FN2O'
period = 'complete_period'
start_extent = '2024-03-01'
end_extent = '2024-08-21'

plt.rcParams.update({'font.size': 12})  # Set global font size
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(14, 7.5), gridspec_kw={'hspace': 0.025})
plot_flux_timeseries(df_agg_filtered, gas=flux, ax=axes[0], period_name=period, start_extent=start_extent, end_extent=end_extent)
plot_cum(df_kg_ha, gas=flux, ax=axes[1], period_name=period, start_extent=start_extent, end_extent=end_extent, mgmt=False)
letterx, lettery = 0.001, 0.95
axes[0].text(letterx, lettery, '(a)', transform=axes[0].transAxes, fontweight='bold', verticalalignment='top')
axes[1].text(letterx, lettery, '(b)', transform=axes[1].transAxes, fontweight='bold', verticalalignment='top')
axes[0].set_ylabel('N$_2$O flux\n(nmol m$^{-2}$ s$^{-1}$)')
axes[1].set_ylabel('Cumulative N$_{2}$O flux\n(kg N$_{2}$O-N ha$^{-1}$)')
axes[1].get_legend().remove()
# Move the legend slightly left
axes[0].legend(loc='upper right', bbox_to_anchor=(0.85, 1), frameon=False)
fig.align_ylabels()
fig.savefig(f'../figures/Timeseries+cumulative_{flux}_{period}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
flux = 'FN2O'
period = 'complete_period'
start_extent = '2024-03-01'
end_extent = '2024-08-21'

plt.rcParams.update({'font.size': 12})  # Set global font size
fig, ax = plt.subplots(figsize=(14, 7.5))
plot_flux_timeseries(df_agg_filtered, gas=flux, ax=ax, period_name=period, start_extent=start_extent, end_extent=end_extent)
ax.set_ylabel('N$_2$O flux\n(nmol m$^{-2}$ s$^{-1}$)')
# Move the legend slightly left
ax.legend(loc='upper right', bbox_to_anchor=(0.8, 1), frameon=False)
fig.align_ylabels()
fig.savefig(f'../figures/Timeseries_{flux}_{period}.png', dpi=300, bbox_inches='tight')
plt.show()